# 05 — Differential Expression & Marker Genes

This notebook covers:
- Ranking marker genes per cluster (Wilcoxon / t-test)
- Dot plots, heatmaps, violin plots of top markers
- Exporting marker gene tables
- UMAP expression overlays


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import config as cfg
from utils import load_adata, save_adata, ensure_dirs

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white')
%matplotlib inline
ensure_dirs(cfg.RESULTS_DIR)
print('Setup complete ✓')

In [ ]:
adata = load_adata(cfg.ADATA_CLUSTERED_PATH)
de = cfg.DIFF_EXPR
print(adata)

## 1. Rank Genes per Cluster

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby=de['groupby'],
    method=de['method'],
    use_raw=de['use_raw'],
    key_added=de['key_added'],
    n_genes=de['n_genes'],
)
print(f'Marker genes ranked using {de["method"]} test')

## 2. Visualise Top Markers

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, key=de['key_added'])

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, key=de['key_added'], groupby=de['groupby'])

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata, n_genes=5, key=de['key_added'], groupby=de['groupby'], show_gene_labels=True
)

In [ ]:
sc.pl.rank_genes_groups_violin(adata, n_genes=3, key=de['key_added'])

## 3. Export Tables

In [ ]:
groups = adata.obs[de['groupby']].cat.categories.tolist()
dfs = []
for g in groups:
    df = sc.get.rank_genes_groups_df(adata, group=str(g), key=de['key_added'])
    df.insert(0, 'cluster', str(g))
    dfs.append(df)

all_markers = pd.concat(dfs, ignore_index=True)
top_markers = all_markers.groupby('cluster').head(de['n_genes']).reset_index(drop=True)

path_all = os.path.join(cfg.RESULTS_DIR, 'all_cluster_markers.csv')
path_top = os.path.join(cfg.RESULTS_DIR, f'top{de["n_genes"]}_markers_per_cluster.csv')
all_markers.to_csv(path_all, index=False)
top_markers.to_csv(path_top, index=False)

print(f'Saved: {path_all}')
print(f'Saved: {path_top}')
top_markers.head(20)

## 4. UMAP Expression Overlay

In [ ]:
# Plot top marker of first 6 clusters on UMAP
top_genes = []
for g in groups[:6]:
    df = sc.get.rank_genes_groups_df(adata, group=str(g), key=de['key_added'])
    gene = df.iloc[0]['names']
    if gene in adata.var_names:
        top_genes.append(gene)

sc.pl.umap(adata, color=top_genes, ncols=3, use_raw=False)

## 5. Save Final AnnData

In [ ]:
save_adata(adata, cfg.ADATA_FINAL_PATH)
print('Done ✓')
print(f'Final AnnData: {cfg.ADATA_FINAL_PATH}')